# Qualia Transmission Protocol (QTP v1)

Welcome to **Qualia** in JupyterLite! This notebook demonstrates how `qualia` decouples embodied sensorimotor peripherals (such as Web Bluetooth robots, range sensors, and cameras) from a local decision runtime speaking QTP over HTTP/JSON.

### Core Concepts
- **`Percept`**: A 2D RGB sensory observation synthesized from point or range sensors.
- **`Retina`**: An egocentric projector that translates 1D range and color samples plus ego-motion into 2D retinal frames with synthetic lateral (`drift`) and looming (`rise`) optical flow.
- **`Consequence`**: The physical transition outcome of the preceding action (`accepted`, `blocked`, `completed`, `progress`, `range`).
- **`Intention`**: The selected discrete action (`"left"`, `"right"`, `"forward"`, or `"still"`), candidate pool readouts (`active`, `candidates`, `scores`), and lateral `.bias` in `[-1.0, +1.0]`.

For the local HTTP walkthrough, run `pixi run reference` and `pixi run lite` in separate terminals, then open this notebook at `http://127.0.0.1:8000/lab/index.html?path=qualia_walkthrough.ipynb`. The reference task allows both this origin and `http://localhost:8000`.

Run the notebook from top to bottom once per fresh reference session. Restart `pixi run reference` before beginning it again; restarting a notebook client does not reset the server. For an offline demonstration, choose `MODE = "local"` in the configuration below. Hosted notebook origins need an explicit additional `--origin` on the reference server.


In [ ]:
try:
    import qualia
except ImportError:
    %pip install -q qualia
    import qualia

print(f"Qualia version : {qualia.__version__}")
print(f"Protocol       : {qualia.QTP_VERSION}")

# Explicit transport choice: "remote" exercises HTTP; "local" runs offline.
MODE = "remote"
URL = qualia.DEFAULT_URL


## 1. Egocentric Retinal Projection (`qualia.Retina`)

Mobile robots often carry a single forward-facing range and color sensor rather than a full 2D camera. `qualia.Retina` maintains an internal angular buffer across a 64×48 canvas:
- Turning **left** shifts previously observed columns **rightward** across the visual field.
- Turning **right** shifts previously observed columns **leftward**.
- Approaching an obstacle (decreasing distance) vertically expands and brightens the central foveal projection, generating synthetic looming cues.

The buffer retains earlier observations. Use `retina.blank()` to remove a cue without retaining its decaying image. A neutral phase annotation alone does not erase the buffer; `cue` annotations never draw patterns.


In [ ]:
retina = qualia.Retina(units="physical", max_distance=100.0)

# Observe a surface at 40 cm with a warm amber color
frame = retina.observe(
    distance=40.0,
    color=(240, 180, 60),
    turn="still"
)

# Evaluating a Percept in a notebook renders a crisp, upscaled 256x192 PNG preview inline!
frame

## 2. Stepping the Sensorimotor Loop (`qualia.Client`)

The default `MODE = "remote"` connects to the reference server at `http://127.0.0.1:8001`. It raises errors if that service is unavailable. `MODE = "local"` explicitly chooses an offline demonstration; both policies have synthetic pool readouts. A remote response's `fallback=False` identifies HTTP transport, while the readiness response identifies the reference policy as `reflex`.


In [ ]:
client = qualia.connect(URL, mode=MODE)
status = await client.status()
print(f"Transport      : {status['engine']}")
print(f"Policy         : {status.get('policy', 'reflex')}")

# Step the loop with the initial Percept
intent = await client.step(frame)

print(f"Action         : {intent.action}")
print(f"Fallback Mode  : {intent.fallback} (True when running local reflex engine)")
print(f"Dome Head Bias : {intent.bias:+.3f} ([-1.0, +1.0] lateral balance)")
print(f"Active Units   : {intent.active}")
print(f"Readout Scores : {intent.scores}")

## 3. Simulating Closed-Loop Trajectories & Obstacle Recovery

Next, let's simulate an embodied trajectory where the robot drives forward toward an obstacle until forward motion is blocked, triggering turn-recovery behavior:

This continues the client and retina from the preceding example, preserving the server's step order.


In [ ]:
# Continue the same session: client resets do not reset a remote service.
last_action = "still"
dist = 60.0
consequence = None
recorded_frames = []

for step_idx in range(10):
    # Generate retinal percept based on current distance and previous action
    percept = retina.observe(distance=dist, color=(220, 100, 100), turn=last_action)
    recorded_frames.append(percept)
    
    # Transmit to client
    intent = await client.step(percept, consequence)
    
    # Determine physical outcome
    blocked = dist <= 15.0 and intent.action == "forward"
    if intent.action == "forward" and not blocked:
        dist = max(5.0, dist - 15.0)
    elif intent.action in ("left", "right"):
        dist = min(80.0, dist + 10.0)
        
    consequence = qualia.Consequence(
        accepted=not blocked,
        blocked=blocked,
        completed=dist <= 8.0,
        progress=0.15 if (intent.action == "forward" and not blocked) else 0.0,
        range=dist / 100.0
    )
    
    print(f"Step {step_idx:2d} | dist={dist:4.1f}cm | action={intent.action:7s} | bias={intent.bias:+.2f} | blocked={blocked} | streak={client.local.streak}")
    last_action = intent.action if not blocked else "still"

### Final Retinal Frame from Trajectory
Here is the view from the final step of the run:

In [ ]:
recorded_frames[-1]

## 4. Web Bluetooth Robot Integration (`ipymovehub`)

When connected to a physical Bluetooth peripheral (like a LEGO Boost Move Hub R2-D2) in JupyterLite, the loop looks like this:

```python
from ipymovehub import MoveHub
hub = MoveHub()
await hub.connect()

# Inside your stepping loop:
dist_cm = hub.distance
percept = retina.observe(distance=dist_cm, color=hub.color_rgb, turn=last_action)
intent = await client.step(percept, consequence)

# Orient the dome head proportionally to lateral candidate pool balance
await hub.turn_head(int(intent.bias * 45))

# Execute discrete drive motor action
await hub.move(intent.action)
```